In [ ]:
### GENERAL SETUP
%matplotlib inline  
# this enables plotting within notebook

#import modules
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
import numpy as np   # basic math library  you will type np.$STUFF  e.g., np.cos(1)
import numpy.linalg as LA
from matplotlib.gridspec import GridSpec
import timeit
import cartopy.crs as ccrs
import datetime
import scipy.stats as stats # imports stats functions https://docs.scipy.org/doc/scipy/reference/stats.html
import cartopy.feature as cfeature
import cftime
from scipy.stats import linregress

import gsw

In [ ]:
def detrend_linear(dat, dim, order):
    """ linear detrend dat along the axis dim """
    params = dat.polyfit(dim=dim, deg=order)
    fit = xr.polyval(dat[dim], params.polyfit_coefficients)
    dat = dat-fit
    return dat

In [ ]:
# open the data!
ds = xr.open_dataset('/glade/work/smogen/SMYLE-extremes/FOSI/SSH.monthly.surface.regrid.nc')['SSH']
ds['time'] = pd.date_range("1958-01", "2020-12", freq="MS")
# center this bad boy on the Pacific
ds = ds.roll(lon=180,roll_coords=True)
ds['lon'] = np.arange(0.5,360.5,1)
# select the NE Pacific
ds = ds.sel(lat=slice(20,62),lon=slice(180,250))
ds = ds/100
#
ssh_fosi = ds

In [ ]:
# # surface
depth = '0m'
# 250 = 23
# 0 = 0
ph = xr.open_dataset('pH_3D.regrid.300m.nc')['pH_3D'].isel(z_t=0); ph2 = 10**(-ph); del ph# ph2 = ph2.isel(z_t=0)
ph2 = ph2.where(ph2 < 1)
o2 = xr.open_dataset('O2.regrid.300m.nc')['O2']; o2 = o2.isel(z_t=0)
no3 = xr.open_dataset('NO3.regrid.300m.nc')['NO3']; no3 = no3.isel(z_t=0)
temp = xr.open_dataset('TEMP.regrid.300m.nc')['TEMP'].isel(z_t=0); 
ssh = xr.open_dataset('SSH.regrid.nc')['SSH'];

In [ ]:
temp['time'] =  pd.date_range("1958-01", "2020-12", freq="MS")
ssh['time'] =  pd.date_range("1958-01", "2020-12", freq="MS")
ph2['time'] =  pd.date_range("1958-01", "2020-12", freq="MS")
o2['time'] =  pd.date_range("1958-01", "2020-12", freq="MS")
no3['time'] =  pd.date_range("1958-01", "2020-12", freq="MS")

## calculate correlations

In [ ]:
from scipy.stats import linregress

def process(ds, order):
    ds_proc = ds.groupby('time.month') - ds.groupby('time.month').mean()
    ds_proc = detrend_linear(ds_proc,'time',order)
    return ds_proc

In [ ]:
# indicators
lat_core = xr.open_dataset('FOSI.Lat.Core.nc')['core']
uvel_str = xr.open_dataset('FOSI.UVEL.strength.nc')['strength']
lat_bifu = xr.open_dataset('FOSI.218.228.bifurcation.nc')

In [ ]:
# process the BGC data!
ph2_proc = process(ph2.where(ph2 < 1), 2)
o2_proc  = process(o2, 1)
no3_proc = process(no3, 1)
temp_proc = process(temp, 1)
ssh_proc = process(ssh, 0)

In [ ]:
# preservation!

def apply_lin_regressm(index, variable):
    range = np.arange(index.min(), index.max(), (index.max() - index.min()) / index.size)
    m, b, r, p, err = linregress(index, variable)
    return m

def m_calc(da, x, coord='time'):
    """Finds the correlation along a given dimension of a dataarray."""

    return xr.apply_ufunc(apply_lin_regressm, 
                          da, 
                          x,
                          input_core_dims=[[coord],[coord]] , 
                          output_core_dims=[[]],
                          vectorize=True,
                          output_dtypes=[float, float]
                          )
    
def apply_lin_regressb(index, variable):
    range = np.arange(index.min(), index.max(), (index.max() - index.min()) / index.size)
    m, b, r, p, err = linregress(index, variable)
    return b

def b_calc(da, x, coord='time'):
    """Finds the correlation along a given dimension of a dataarray."""

    return xr.apply_ufunc(apply_lin_regressb, 
                          da, 
                          x,
                          input_core_dims=[[coord],[coord]] , 
                          output_core_dims=[[]],
                          vectorize=True,
                          output_dtypes=[float, float]
                          )

def apply_lin_regressr(index, variable):
    range = np.arange(index.min(), index.max(), (index.max() - index.min()) / index.size)
    m, b, r, p, err = linregress(index, variable)
    return r

def r_calc(da, x, coord='time'):
    """Finds the correlation along a given dimension of a dataarray."""

    return xr.apply_ufunc(apply_lin_regressr, 
                          da, 
                          x,
                          input_core_dims=[[coord],[coord]] , 
                          output_core_dims=[[]],
                          vectorize=True,
                          output_dtypes=[float, float]
                          )

def apply_lin_regressp(index, variable):
    range = np.arange(index.min(), index.max(), (index.max() - index.min()) / index.size)
    m, b, r, p, err = linregress(index, variable)
    return p

def p_calc(da, x, coord='time'):
    """Finds the correlation along a given dimension of a dataarray."""

    return xr.apply_ufunc(apply_lin_regressp, 
                          da, 
                          x,
                          input_core_dims=[[coord],[coord]] , 
                          output_core_dims=[[]],
                          vectorize=True,
                          output_dtypes=[float, float]
                          )

In [ ]:
## roll both index and data

## Calculate data

rolling_length = 6
rolling_length2 = 6

index_to_compare = lat_core # npgo_str, uvel_str, lat_core, lat_bifu
index_name = 'lat_core' # npgo, uvel, lat_core, lat_bifu

variables    = [ph2_proc, o2_proc, no3_proc, bio_proc, npp_proc, chl_proc, salt_proc, spic_proc]
variable_str = ['pH', 'O2', 'NO3', 'Bio', 'NPP', 'Chl', 'SALT', 'spiciness']

tmpb = []
tmpm = []
tmpr = []
tmpp = []

for i in range(len(variables)):
    m = m_calc(index_to_compare.rolling(time=rolling_length2,center=True).mean().sel(time=slice('1960-01','2019-12')), variables[i].rolling(time=rolling_length,center=True).mean().sel(time=slice('1960-01','2019-12')))
    m.expand_dims('data_var')
    m['data_var'] = variable_str[i]
    tmpm.append(m)

    b = b_calc(index_to_compare.rolling(time=rolling_length2,center=True).mean().sel(time=slice('1960-01','2019-12')), variables[i].rolling(time=rolling_length,center=True).mean().sel(time=slice('1960-01','2019-12')))
    b.expand_dims('data_var')
    b['data_var'] = variable_str[i]
    tmpb.append(b)

    r = r_calc(index_to_compare.rolling(time=rolling_length2,center=True).mean().sel(time=slice('1960-01','2019-12')), variables[i].rolling(time=rolling_length,center=True).mean().sel(time=slice('1960-01','2019-12')))
    r.expand_dims('data_var')
    r['data_var'] = variable_str[i]
    tmpr.append(r)

    p = p_calc(index_to_compare.rolling(time=rolling_length2,center=True).mean().sel(time=slice('1960-01','2019-12')), variables[i].rolling(time=rolling_length,center=True).mean().sel(time=slice('1960-01','2019-12')))
    p.expand_dims('data_var')
    p['data_var'] = variable_str[i]
    tmpp.append(p)
    
m_npgo = xr.concat(tmpm,dim='data_var'); m_npgo = m_npgo.to_dataset(name = 'm')
b_npgo = xr.concat(tmpb,dim='data_var'); b_npgo = b_npgo.to_dataset(name = 'b')
r_npgo = xr.concat(tmpr,dim='data_var'); r_npgo = r_npgo.to_dataset(name = 'r')
p_npgo = xr.concat(tmpp,dim='data_var'); p_npgo = p_npgo.to_dataset(name = 'p')

combine = xr.merge([m_npgo, b_npgo, r_npgo, p_npgo])

In [ ]:
combine.to_netcdf(index_name + '.' + depth + '.' + str(rolling_length) + 'month.index.nc')